# HW 8 — Bot Classifier: Zero-Shot Evaluation + Fine-Tuning
**Run on GPU** (Runtime → Change runtime type → T4 GPU)

Steps:
1. Upload `data.zip` from Kaggle → Run dataset setup
2. Run zero-shot evaluation
3. Run fine-tuning
4. Download the trained model → place in `hw_8/model_checkpoint/best/`

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets scikit-learn pandas kagglehub accelerate torch

In [ ]:
# 2. Upload dataset files (upload data.zip from Kaggle, or individual files)
#    The zip contains: train.json, ytrain.csv, test.json, ytest.csv
from google.colab import files
import zipfile, os, json

uploaded = files.upload()

os.makedirs("data", exist_ok=True)
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('data')
    else:
        os.rename(fname, f"data/{fname}")

print("Files in data/:", os.listdir("data"))

# Quick peek
with open("data/train.json") as f:
    dialogs = json.load(f)
import pandas as pd
labels = pd.read_csv("data/ytrain.csv")
print(f"Dialogs: {len(dialogs)}, Labels: {len(labels)}")
print(f"Total messages: {sum(len(msgs) for msgs in dialogs.values())}")
print(f"Bot ratio: {labels['is_bot'].mean():.4f}")

In [ ]:
# 3. Load data into helper structures
import json
import numpy as np
import pandas as pd
from pathlib import Path

def load_data(data_dir="data"):
    with open(f"{data_dir}/train.json") as f:
        dialogs = json.load(f)
    labels_df = pd.read_csv(f"{data_dir}/ytrain.csv")
    label_map = {}
    for _, row in labels_df.iterrows():
        label_map[(row["dialog_id"], row["participant_index"])] = int(row["is_bot"])
    return dialogs, label_map

dialogs, label_map = load_data()
print(f"Loaded {len(dialogs)} dialogs, {len(label_map)} participant-level labels")

## Part A: Zero-Shot Classification Evaluation

In [ ]:
# 4. Run zero-shot NLI classification on ALL training data (GPU-accelerated)
import time
import torch
from transformers import pipeline
from tqdm.notebook import tqdm
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)

MODEL_NAME = "typeform/distilbert-base-uncased-mnli"
CANDIDATE_LABELS = ["bot", "human"]
HYPOTHESIS_TEMPLATE = "This message was written by a {}."

device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU' if device == 0 else 'CPU'}")

print(f"Loading zero-shot classifier: {MODEL_NAME} ...")
start = time.time()
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=device,
)
print(f"Loaded in {time.time() - start:.1f}s")

In [ ]:
# 5. Classify participants (aggregate per-message probabilities)
participant_texts = {}
for dialog_id, messages in dialogs.items():
    for msg in messages:
        key = (dialog_id, msg["participant_index"])
        participant_texts.setdefault(key, []).append(msg["text"])

print(f"Classifying {len(participant_texts)} participants...")

y_true, y_prob = [], []
for (dialog_id, pidx), texts in tqdm(participant_texts.items(), desc="Zero-shot classifying"):
    key = (dialog_id, pidx)
    if key not in label_map:
        continue
    probs = []
    for text in texts:
        if not text.strip():
            probs.append(0.5)
            continue
        result = classifier(text, candidate_labels=CANDIDATE_LABELS, hypothesis_template=HYPOTHESIS_TEMPLATE)
        bot_idx = result["labels"].index("bot")
        probs.append(result["scores"][bot_idx])
    y_true.append(label_map[key])
    y_prob.append(float(np.mean(probs)))

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

In [ ]:
# 6. Zero-shot evaluation metrics
print("=" * 60)
print("ZERO-SHOT CLASSIFICATION RESULTS")
print(f"Model: {MODEL_NAME}")
print(f"Participants evaluated: {len(y_true)}")
print(f"Bot ratio (data):      {y_true.mean():.4f}")
print(f"Bot ratio (predicted): {y_pred.mean():.4f}")
print()
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}")
print()
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(f"  TN={cm[0][0]:5d}  FP={cm[0][1]:5d}")
print(f"  FN={cm[1][0]:5d}  TP={cm[1][1]:5d}")
print()
print(classification_report(y_true, y_pred, target_names=["human", "bot"], zero_division=0))
print("=" * 60)

## Part B: Fine-Tune DistilBERT Classifier

In [ ]:
# 7. Build per-message dataset (each message gets its participant's label)
from datasets import Dataset

texts, labels = [], []
for dialog_id, messages in dialogs.items():
    for msg in messages:
        key = (dialog_id, msg["participant_index"])
        if key in label_map:
            texts.append(msg["text"])
            labels.append(label_map[key])

print(f"Total messages: {len(texts)}, bot ratio: {sum(labels)/len(labels):.4f}")

dataset = Dataset.from_dict({"text": texts, "label": labels})
split = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]
eval_ds = split["test"]
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

In [ ]:
# 8. Tokenize the datasets
from transformers import AutoTokenizer

BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(preprocess, batched=True)
eval_ds = eval_ds.map(preprocess, batched=True)
print("Tokenization done.")

In [ ]:
# 9. Load model and set up training
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from transformers.trainer_utils import EvalPrediction

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, probs),
    }

training_args = TrainingArguments(
    output_dir="model_checkpoint",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training on", "GPU" if torch.cuda.is_available() else "CPU")

In [ ]:
# 10. Train!
import time
start = time.time()
trainer.train()
print(f"Training completed in {(time.time() - start) / 60:.1f} minutes")

In [ ]:
# 11. Final evaluation
results = trainer.evaluate()
print("\n" + "=" * 60)
print("FINE-TUNED MODEL RESULTS")
print(f"Model: {BASE_MODEL}")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")
print("=" * 60)

In [ ]:
# 12. Save best model
BEST_DIR = "model_checkpoint/best"
trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
print(f"Model saved to {BEST_DIR}/")
print("Files:", os.listdir(BEST_DIR))

In [ ]:
# 13. Quick smoke test
from transformers import pipeline
pipe = pipeline("text-classification", model=BEST_DIR, device=device)
test_texts = ["Hi, how are you?", "I am an AI language model here to help you"]
for t in test_texts:
    r = pipe(t)[0]
    print(f"'{t[:50]}...' → label={r['label']}, score={r['score']:.4f}")

In [ ]:
# 14. Download the model
!zip -r bot_classifier_model.zip model_checkpoint/best/
from google.colab import files
files.download("bot_classifier_model.zip")

## After downloading:

```bash
# Unzip the model into your hw_8 directory
unzip -o bot_classifier_model.zip -d /Users/tom/Desktop/mlops/hw_8/

# Then start the server:
cd /Users/tom/Desktop/mlops/hw_8
uv run python3 server.py
```